In [18]:
import mpbn
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import os
import re
import json
import itertools
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS

models_dir = Path("/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/ensemble")
data_file = "/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/data_binarized_tf_gene_sep.csv"
data = pd.read_csv(data_file,index_col=0)

In [19]:
def screen_models(models_dir, data, gene, value, direction):
    """
    Screen Boolean models after a KO/KI perturbation.

    Parameters
    ----------
    models_dir : Path
        Folder containing .bnet models.
    data : pd.DataFrame
        Binarized macrostates.
    gene : str
        Gene to perturb.
    value : str
        "0" for KO, "1" for KI.

    Returns
    -------
    selected_models : list
        List of selected model names.
    """

    selected_models = []

    for model_file in models_dir.glob("*.bnet"):

        try:
            # Load model
            bn = mpbn.MPBooleanNetwork(str(model_file))

            # Perturbation
            bn[gene] = value

            # Compute attractors
            attractors = pd.DataFrame(list(bn.attractors()))

            if attractors.empty:
                continue

            # Keep common genes
            common_genes = attractors.columns.intersection(data.columns)

            if len(common_genes) == 0:
                continue

            attractors = attractors[common_genes]
            data_subset = data[common_genes]

            attractors = attractors[data_subset.columns]
            attractors.index = [f"A{i}" for i in range(len(attractors))]

            # Similarity matrix
            similarity = pd.DataFrame(index=attractors.index)

            for state in data_subset.index:
                row = data_subset.loc[state]
                mask = row.notna()

                similarity[state] = (
                    attractors.loc[:, mask]
                    .eq(row[mask], axis=1)
                    .mean(axis=1)
                )

            # Selection criterion
            if direction == "KO":
                cond1 = (similarity["S3"] > similarity["S2"]).all()
                cond2 = similarity["S3"].max() > 0.99

            elif direction == "KI":
                cond1 = (similarity["S2"] > similarity["S3"]).all()
                cond2 = similarity["S2"].max() > 0.99

            else:
                raise ValueError("direction must be 'KO' or 'KI'")

            if cond1 and cond2:
                selected_models.append(model_file.name)

        except Exception as e:
            print(model_file.name, e)

    return selected_models

In [20]:
data = pd.read_csv(data_file, index_col=0)

selected_models_ko = screen_models(models_dir=models_dir,data=data,gene="CEBPA",value="0",direction="KO")

selected_models_ki = screen_models(models_dir=models_dir,data=data,gene="CEBPA",value="1",direction="KI")

#Intersection KO and Ki 
models_cebpa = list(set(selected_models_ko) & set(selected_models_ki))

print(f"KO : {len(selected_models_ko)}")
print(f"KI : {len(selected_models_ki)}")
print(f"Intersection : {len(models_cebpa)}")

KO : 178
KI : 182
Intersection : 178


Sort the templates and place the selected ones in a folder

In [41]:
# New folder
selected_dir = models_dir / "models_selected"
selected_dir.mkdir(exist_ok=True)

# Copy models selected
for model_name in models_cebpa:
    src = models_dir / model_name
    dst = selected_dir / model_name

    if src.exists():
        shutil.copy2(src, dst)
    else:
        print(f"File no found : {model_name}")

print(f"{len(models_cebpa)} modeles paste : {selected_dir}")

178 modeles paste : /home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/ensemble/models_selected


Teste

In [27]:
# LOAD MODELS
def parse_bnet_file(path):
    model = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.lower().replace(" ", "").startswith("targets,factors"):
                continue
            if "," in line:
                node, formula = line.split(",", 1)
            else:
                node, formula = re.split(r"\s+", line, maxsplit=1)
            node = node.strip()
            formula = re.sub(r"\s+", " ", formula.strip())
            model[node] = formula
    return model


def load_models(models_dir="bnet_models", pattern=".bnet"):
    if not os.path.isdir(models_dir):
        raise FileNotFoundError(
            f"Dossier '{models_dir}' introuvable. Modifie `models_dir` dans "
            f"load_models() pour pointer vers tes fichiers .bnet."
        )
    files = sorted(f for f in os.listdir(models_dir) if f.endswith(pattern))
    if not files:
        raise FileNotFoundError(f"Aucun fichier {pattern} trouvé dans '{models_dir}'.")

    models, model_names = [], []
    for fname in files:
        models.append(parse_bnet_file(os.path.join(models_dir, fname)))
        model_names.append(fname)

    print(f"{len(models)} fichiers .bnet chargés depuis '{models_dir}'.")
    node_sets = [frozenset(m.keys()) for m in models]
    if len(set(node_sets)) > 1:
        print("⚠️ Please note: some models do not have exactly the same nodes " )
    return models, model_names


# BUILD MATRIX
def build_rule_matrix(models):
    nodes = sorted(set().union(*[m.keys() for m in models]))
    n_models = len(models)
    df = pd.DataFrame(index=nodes, columns=range(n_models), dtype=object)
    for j, m in enumerate(models):
        for node in nodes:
            df.at[node, j] = m.get(node, None)
    return df


# FILTER IDENTIC NODES
def filter_variable_nodes(rule_df):
    identical_nodes = []
    variable_nodes = []
    for node in rule_df.index:
        n_unique = pd.Series(rule_df.loc[node]).astype(str).nunique()
        if n_unique <= 1:
            identical_nodes.append(node)
        else:
            variable_nodes.append(node)

    filtered_df = rule_df.loc[variable_nodes]
    return filtered_df, identical_nodes

def node_diversity_summary(rule_df):
    rows = []
    for node in rule_df.index:
        rules = rule_df.loc[node].tolist()
        counts = pd.Series(rules).value_counts()
        n_unique = len(counts)
        majority_rule = counts.index[0]
        majority_frac = counts.iloc[0] / len(rules)
        rows.append({
            "node": node,
            "n_unique_rules": n_unique,
            "majority_rule": majority_rule,
            "majority_fraction": round(majority_frac, 3),
        })
    return pd.DataFrame(rows).sort_values("n_unique_rules", ascending=False)


# HEATMAP nodes x modeles
def plot_rule_heatmap(rule_df, outpath):
    encoded = rule_df.copy()
    for node in rule_df.index:
        uniques = pd.Series(rule_df.loc[node]).astype(str)
        codes, _ = pd.factorize(uniques)
        encoded.loc[node] = codes

    fig, ax = plt.subplots(figsize=(min(20, encoded.shape[1] * 0.05 + 4), max(6, len(encoded) * 0.3)))
    im = ax.imshow(encoded.values.astype(float), aspect="auto", cmap="tab20")
    ax.set_yticks(range(len(encoded.index)))
    ax.set_yticklabels(encoded.index, fontsize=14)
    ax.set_xlabel(f"Models (n={encoded.shape[1]})")
    ax.set_title("Rule variant used per template and per node\n"
                  "(one colour = one different logical formula)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


# Models CLUSTERING
def model_distance_matrix(rule_df):
    n_models = rule_df.shape[1]
    dist = np.zeros((n_models, n_models))
    values = rule_df.values
    for i, j in itertools.combinations(range(n_models), 2):
        d = np.sum(values[:, i] != values[:, j])
        dist[i, j] = dist[j, i] = d
    return dist


def plot_model_clustering(dist_matrix, outpath, n_show_clusters=6):
    condensed = squareform(dist_matrix, checks=False)
    Z = linkage(condensed, method="average")

    fig, ax = plt.subplots(figsize=(max(8, dist_matrix.shape[0] * 0.08), 5))
    dendrogram(Z, ax=ax, no_labels=(dist_matrix.shape[0] > 40))
    ax.set_title("Hierarchical clustering of models\n"
                  "(similarity = few differences in the rules)")
    ax.set_ylabel("Distance (number of nodes with a different rule)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

    clusters = fcluster(Z, t=n_show_clusters, criterion="maxclust")
    return clusters

# MAIN
def main():
    models, model_names = load_models(models_dir="/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/ensemble/models_selected")

    rule_df_full = build_rule_matrix(models)

    rule_df, identical_nodes = filter_variable_nodes(rule_df_full)
    print(f"\n{len(identical_nodes)} identical node(s) in all models, removed from the pins :")
    print(identical_nodes if identical_nodes else "(None)")
    print(f"{len(rule_df)} variable node(s) retained for the analysis.")

    with open(os.path.join(OUTDIR, "identical_nodes_removed.txt"), "w") as f:
        f.write("\n".join(identical_nodes))

    summary = node_diversity_summary(rule_df)
    summary.to_csv(os.path.join(OUTDIR, "node_diversity_summary.csv"), index=False)
    print("\nSummary by node variability (sorted from most variable to most consensual) :")
    print(summary.to_string(index=False))

    plot_rule_heatmap(rule_df, os.path.join(OUTDIR, "rule_heatmap.png"))

    dist_matrix = model_distance_matrix(rule_df)
    clusters = plot_model_clustering(dist_matrix, os.path.join(OUTDIR, "model_clustering.png"))
    plot_model_mds(dist_matrix, clusters, os.path.join(OUTDIR, "model_mds_projection.png"))

    print(f"\nFigures and abstract written in : {OUTDIR}/")


if __name__ == "__main__":
    main()

178 fichiers .bnet chargés depuis '/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/ensemble/models_selected'.

361 identical node(s) in all models, removed from the pins :
['ABCA1', 'ABCB1', 'ABCB4', 'ACOT7', 'ACSL1', 'ACTG1', 'ACYP2', 'ADAM10', 'AFG1L', 'ALDH1A1', 'ALDH5A1', 'ALOX5', 'AMPD3', 'ANGPT2', 'APC', 'APOC2', 'APOE', 'APPBP2', 'ARNT', 'ATF3', 'ATF6', 'ATG10', 'ATG4C', 'ATM', 'ATP2A3', 'ATP2B4', 'ATXN10', 'B4GALT1', 'BACH2', 'BANF1', 'BCAS3', 'BCL11A', 'BCL2A1', 'BHLHE41', 'BICD2', 'BID', 'BIRC3', 'BLM', 'BRCA1', 'BRCA2', 'BTD', 'BTG3', 'BTRC', 'C1QBP', 'CALR', 'CASP1', 'CASP8AP2', 'CAV1', 'CBLB', 'CCDC7', 'CCL3', 'CCL4', 'CCM2', 'CCND3', 'CCR1', 'CCT5', 'CD14', 'CD24', 'CD2AP', 'CD40', 'CD58', 'CD63', 'CD68', 'CD72', 'CD9', 'CDC25B', 'CDKN1A', 'CDKN1C', 'CDKN2D', 'CDX2', 'CEACAM1', 'CEBPB', 'CEBPE', 'CEBPG', 'CETP', 'CFLAR', 'CHI3L1', 'CHST11', 'CKS2', 'CNR1', 'CRABP2', 'CREB1', 'CREG1', 'CSTA', 'CSTB', 'CTCF', 'CTSB', 'CTSD', 'CTSL', 

/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/sklearn/manifold/_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(



Figures and abstract written in : boolean_models_diversity_plots/
